# 🏥 K-Means Clustering — Patient Risk Segmentation

**Biomedical ML Toolkit** | C. Sathish Kumar

Group patients into Low/Medium/High risk clusters using K-Means.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
print('Libraries loaded!')

In [ ]:
try:
    df = pd.read_csv('../datasets/diabetes.csv')
except:
    url = 'https://raw.githubusercontent.com/YOUR_USERNAME/biomedical-ml-toolkit/main/datasets/diabetes.csv'
    df = pd.read_csv(url)

features = ['Glucose','BMI','Age','BloodPressure','Insulin']
X = df[features].copy()
for col in features:
    X[col] = X[col].replace(0, X[col].mean())
print(f'{len(X)} patients loaded')

In [ ]:
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

# Elbow Method
wcss = []
for k in range(2,9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_s)
    wcss.append(km.inertia_)

plt.figure(figsize=(8,4))
plt.plot(range(2,9), wcss, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Inertia)')
plt.title('Elbow Method — Finding Optimal K')
plt.xticks(range(2,9))
plt.grid(alpha=0.3)
plt.show()
print('Look for the elbow point in the graph above!')

In [ ]:
# Final clustering with K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_s) + 1

# Show cluster summary
for c in sorted(df['Cluster'].unique()):
    grp = df[df['Cluster']==c]
    print(f'\nCluster {c}: {len(grp)} patients')
    print(f'  Avg Glucose: {grp["Glucose"].mean():.1f} | Avg BMI: {grp["BMI"].mean():.1f}')
    print(f'  Avg Age: {grp["Age"].mean():.1f} | Diabetic%: {grp["Outcome"].mean()*100:.1f}%')

In [ ]:
# Scatter plot: Glucose vs BMI coloured by cluster
plt.figure(figsize=(8,6))
colors = {1:'#4CAF50', 2:'#FF9800', 3:'#F44336'}
labels_map = {1:'Low Risk', 2:'Medium Risk', 3:'High Risk'}

for c in sorted(df['Cluster'].unique()):
    mask = df['Cluster']==c
    plt.scatter(df[mask]['Glucose'], df[mask]['BMI'],
                c=colors[c], label=labels_map[c], alpha=0.6, s=40)

plt.xlabel('Glucose (mg/dL)', fontsize=12)
plt.ylabel('BMI', fontsize=12)
plt.title('Patient Risk Segmentation (K-Means, K=3)', fontsize=13)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.show()